In [11]:
!pip install -q transformers datasets accelerate torch

In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer, get_scheduler
from torch.optim import AdamW
from datasets import load_dataset
from tqdm.auto import tqdm

# Setup device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [13]:
# Teacher: A larger/better model (DistilGPT2 is actually smaller but serves as a proxy for this demo)
# Student: A tiny version of GPT2
teacher_id = "distilgpt2" 
student_id = "gpt2" # In a real scenario, the student would be much smaller than the teacher

tokenizer = AutoTokenizer.from_pretrained(teacher_id)
tokenizer.pad_token = tokenizer.eos_token

teacher_model = AutoModelForCausalLM.from_pretrained(teacher_id).to(device)
student_model = AutoModelForCausalLM.from_pretrained(student_id).to(device)

# Freeze Teacher weights (we don't train the teacher)
for param in teacher_model.parameters():
    param.requires_grad = False

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [14]:
# Using a small subset of WikiText for demonstration
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train[:5%]")

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True, remove_columns=["text"])
tokenized_datasets.set_format("torch")

train_dataloader = DataLoader(tokenized_datasets, batch_size=8, shuffle=True)

In [15]:
class DistillationLoss(nn.Module):
    def __init__(self, temperature=2.0, alpha=0.5):
        super(DistillationLoss, self).__init__()
        self.temperature = temperature
        self.alpha = alpha
        self.ce_loss = nn.CrossEntropyLoss()
        self.kl_div = nn.KLDivLoss(reduction="batchmean")

    def forward(self, student_logits, teacher_logits, labels):
        # 1. Standard Cross Entropy Loss (Student vs Ground Truth)
        # Shift so that tokens predict the next token
        shift_logits = student_logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()
        loss_ce = self.ce_loss(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))

        # 2. KL Divergence Loss (Student Softmax vs Teacher Softmax)
        # Apply Temperature scaling
        p_student = F.log_softmax(student_logits / self.temperature, dim=-1)
        p_teacher = F.softmax(teacher_logits / self.temperature, dim=-1)
        
        loss_kl = self.kl_div(p_student, p_teacher) * (self.temperature ** 2)

        # 3. Combined Weighted Loss
        return (self.alpha * loss_ce) + ((1 - self.alpha) * loss_kl)

In [16]:
optimizer = AdamW(student_model.parameters(), lr=5e-5)
distill_criterion = DistillationLoss(temperature=2.0, alpha=0.5)

num_epochs = 1
num_training_steps = num_epochs * len(train_dataloader)
lr_scheduler = get_scheduler(
    "linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps
)

In [17]:
student_model.train()
teacher_model.eval()

progress_bar = tqdm(range(num_training_steps))

for epoch in range(num_epochs):
    for batch in train_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        
        # Get Teacher predictions (No gradient needed)
        with torch.no_grad():
            teacher_outputs = teacher_model(**batch)
            teacher_logits = teacher_outputs.logits

        # Get Student predictions
        student_outputs = student_model(**batch)
        student_logits = student_outputs.logits

        # Compute Distillation Loss
        loss = distill_criterion(student_logits, teacher_logits, batch["input_ids"])

        # Backpropagation
        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        
        progress_bar.update(1)
        progress_bar.set_postfix({"loss": loss.item()})

print("Distillation Complete!")


  0%|          | 0/230 [00:00<?, ?it/s]

Distillation Complete!


In [18]:
student_model.save_pretrained("./distilled_student_model")
tokenizer.save_pretrained("./distilled_student_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./distilled_student_model/tokenizer_config.json',
 './distilled_student_model/tokenizer.json')

In [19]:
print("\n--- Generating Output ---")

# Load the distilled model
my_slm = AutoModelForCausalLM.from_pretrained("./distilled_student_model").to(device)
my_tokenizer = AutoTokenizer.from_pretrained("./distilled_student_model")

# Define a prompt
prompt = "Artificial Intelligence is "
input_ids = my_tokenizer.encode(prompt, return_tensors="pt").to(device)

# Generate text
output_sequences = my_slm.generate(
    input_ids=input_ids,
    max_length=50,
    temperature=0.7,
    top_k=50,
    top_p=0.95,
    do_sample=True,
    num_return_sequences=1,
    pad_token_id=my_tokenizer.eos_token_id
)

# Decode and print
generated_text = my_tokenizer.decode(output_sequences[0], skip_special_tokens=True)
print(f"Prompt: {prompt}")
print(f"Generated: {generated_text}")


--- Generating Output ---


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Prompt: Artificial Intelligence is 
Generated: Artificial Intelligence is iced by the same people who used to talk about the concept of "intelligent" AI. It is also known as Artificial Intelligence (AI). The term "intelligent" is an extremely popular term to describe the ability to


In [20]:
import os
import subprocess
import shutil

# 1. Configuration
repo_save_path = "trained_models/my_slm_v1"
zip_filename = "my_slm_v1.zip"

def is_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

# 2. Check if model and tokenizer exist
if 'my_slm' not in globals() or 'my_tokenizer' not in globals():
    print("Error: 'my_slm' or 'my_tokenizer' not found.")
    print("Please make sure you have run the previous cells in the notebook.")
else:
    # 3. Save the model and tokenizer
    os.makedirs(repo_save_path, exist_ok=True)
    print(f"Saving model to {repo_save_path}...")
    my_slm.save_pretrained(repo_save_path)
    my_tokenizer.save_pretrained(repo_save_path)
    print("Model and tokenizer saved successfully in the notebook environment!")

    if is_colab():
        print("\n--- Google Colab Detected ---")
        print("Note: Saving here only keeps files on Google's servers.")
        
        # Zip the folder
        shutil.make_archive("my_slm_v1", 'zip', repo_save_path)
        print(f"Created zip archive: {zip_filename}")
        
        # Trigger download
        try:
            from google.colab import files
            print("Triggering download to your PC...")
            files.download(zip_filename)
        except Exception as e:
            print(f"Download triggered but might need manual check: {e}")
            print("You can manually download it from the 'Files' tab on the left.")
        
        print("\n--- To save directly to GitHub from Colab ---")
        print("1. Create a Personal Access Token on GitHub (classic).")
        print("2. Use: !git remote set-url origin https://<TOKEN>@github.com/<USER>/<REPO>.git")
        print("3. Run: !git add . && !git commit -m 'Add model' && !git push")
    else:
        print("\n--- Local Environment Detected ---")
        # 4. Optional: Stage files with Git
        try:
            print("\n--- Git Operations ---")
            subprocess.run(["git", "add", repo_save_path], check=True)
            print(f"Staged: {repo_save_path}")
            print("Ready to push to GitHub!")
        except Exception as e:
            print(f"Git operations skipped or failed: {e}")
            print("Tip: Use your IDE's Git tools to push to GitHub.")

Saving model to trained_models/my_slm_v1...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer saved successfully!

--- Git Operations ---
Git operations skipped or failed: Command '['git', 'add', 'trained_models/my_slm_v1']' returned non-zero exit status 128.
You can manually run 'git push' from your terminal.
